# Predict the Model's Attention w/Human Data from TED

In [9]:
!newgrp ADAS

(base) ]0;dporres@cudahpc43: /data/121-2/Experiments/dporres/CILv2_multiviewdporres@cudahpc43:/data/121-2/Experiments/dporres/CILv2_multiview$ ^C

(base) ]0;dporres@cudahpc43: /data/121-2/Experiments/dporres/CILv2_multiviewdporres@cudahpc43:/data/121-2/Experiments/dporres/CILv2_multiview$ 

In [11]:
import os

os.chdir('/data/121-2/Experiments/dporres/CILv2_multiview')

In [12]:
# Load the model
# Load the network

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from network.models.architectures.CIL_multiview.CIL_multiview import CIL_multiview
import os
import json
import torch
import torchvision.transforms.functional as TF

from configs import g_conf, set_type_of_process, merge_with_yaml
from _utils.training_utils import check_saved_checkpoints
from _utils import eval_utils
from einops import rearrange

import numpy as np
import matplotlib.pyplot as plt


os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# Load the configuration for the model (folder and experiment name)
exp_batch = 'TED'
exp_name = '00_CIL++_3cam_Town01_17hdata-Unfiltered_noAttention_bs120_400x225'

merge_with_yaml(os.path.join('configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
g_conf.DATASET_PATH = '/datatmp/Datasets/yixiao/CARLA/'

os.environ['DATASET_PATH'] = g_conf.DATASET_PATH

model = CIL_multiview(g_conf.MODEL_CONFIGURATION)


# Get the path to the latest checkpoint (if no ckpt_number is defined)
checkpoint_path = os.path.join('/data/121-2/Experiments/dporres/VisionTFM', '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')

eval_utils.load_model_from_checkpoint(model=model, checkpoint_path=checkpoint_path, checkpoint_number=80)

CIL_multiview(
  (encoder_embedding_perception): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from network.models.architectures.CIL_multiview.CIL_multiview import CIL_multiview
import os
import json
from configs import g_conf, set_type_of_process, merge_with_yaml
from _utils.training_utils import check_saved_checkpoints
import torch
from typing import OrderedDict, Union
from PIL import Image
from einops import rearrange
from dataloaders.transforms import canbus_normalization, train_transform

In [4]:
def open_image(dataset_root_path: Union[str, os.PathLike],
               img_name: str) -> Image.Image:
    """
    Open and return a PIL.Image.Image object from the given image name.
    """
    img = Image.open(os.path.join(dataset_root_path, img_name))
    if 'virtual_attention' in img_name:
        return img.convert('L')
    return img.convert('RGB')

def get_single_data_point(data_root_dir: Union[str, os.PathLike],
                          rgb_img_name: str,
                          canbus_json_name: str,
                          data_number: int, 
                          synth_att_name: str = None,
                          synth_att_types: list = ['']) -> dict:
    """
    
    """

    img_paths_dict = {
        f'{rgb_img_name}_central': [open_image(data_root_dir, f'{rgb_img_name}_central{data_number:06d}.png')],
        f'{rgb_img_name}_left': [open_image(data_root_dir, f'{rgb_img_name}_left{data_number:06d}.png')],
        f'{rgb_img_name}_right': [open_image(data_root_dir, f'{rgb_img_name}_right{data_number:06d}.png')],
    }

    if synth_att_name is not None:
        for synth_att_type in synth_att_types:
            img_paths_dict.update({
                f'{synth_att_name}_central_{synth_att_type}': [open_image(data_root_dir, f'{synth_att_name}_central_{synth_att_type}{data_number:06d}.jpg')],
                f'{synth_att_name}_left_{synth_att_type}': [open_image(data_root_dir, f'{synth_att_name}_left_{synth_att_type}{data_number:06d}.jpg')],
                f'{synth_att_name}_right_{synth_att_type}': [open_image(data_root_dir, f'{synth_att_name}_right_{synth_att_type}{data_number:06d}.jpg')],
            })

    canbus_paths = [os.path.join(data_root_dir, f'{canbus_json_name}{data_number:06d}.json')]

    # Sanity check
    for camera_type, img_paths in img_paths_dict.items():
        if len(img_paths) != len(canbus_paths):
            print(camera_type, len(img_paths), len(canbus_paths))
            raise RuntimeError('The numbers of images and canbus data are not mathced!!')

    full_dataset = []
    for i in range(len(canbus_paths)):
        datapoint = {}
        datapoint['can_bus'] = dict()
        f = open(canbus_paths[i], 'r')
        canbus_data = json.loads(f.read())
        for value in g_conf.TARGETS + g_conf.OTHER_INPUTS:
            datapoint['can_bus'][value] = canbus_data[value]
        datapoint['can_bus'] = canbus_normalization(datapoint['can_bus'], g_conf.DATA_NORMALIZATION)
        for camera_type, img_paths in img_paths_dict.items():
            datapoint[camera_type] = img_paths[i]
        full_dataset.append(datapoint)
    
    return train_transform(full_dataset[0], tuple(g_conf.IMAGE_SHAPE))

In [5]:
dataset_path_root = '/data-net/ted/users'
user_root_path = '/D019/Town01/R01'

data = get_single_data_point(
    dataset_root_path=os.path.join(dataset_path_root, user_root_path, 'RGB'),

SyntaxError: unexpected EOF while parsing (488437997.py, line 5)

In [7]:
from pathlib import Path
from typing import Dict, List, Set
from collections import defaultdict
import logging
from tqdm.auto import tqdm

def find_synchronized_ticks(
    base_path: str, 
    user_num: int,
    town: str = "Town01",
    route: str = "R01",
    weather: str = "ClearNoon"
) -> Dict[int, bool]:
    """
    Find all world_ticks that have complete data (RGB + CAN bus).
    """
    # Setup logging
    logging.basicConfig(
        filename='data_sync_log.txt',
        level=logging.INFO,
        format='%(asctime)s - %(message)s'
    )
    
    ticks_by_sensor = defaultdict(set)
    user_path = Path(base_path) / f"D{user_num:03d}" / town / route
    
    # Create main progress bar for sensor types
    sensor_pbar = tqdm(total=4, desc=f"Scanning sensors (User {user_num:03d})")
    
    # Get CB ticks
    cb_path = user_path / "CB"
    cb_pattern = f"D{user_num:03d}_{town}_{route}_CB_*.json"
    cb_files = list(cb_path.glob(cb_pattern))
    
    for cb_file in tqdm(cb_files, desc="Processing CAN bus data", leave=False):
        world_tick = int(cb_file.stem.split('_')[-1])
        ticks_by_sensor['can_bus'].add(world_tick)
    sensor_pbar.update(1)
    
    # Get RGB ticks for each camera
    for camera in ['Left', 'Central', 'Right']:
        rgb_path = user_path / "RGB" / camera / weather
        rgb_pattern = f"D{user_num:03d}_{town}_{route}_RGB_{camera}_{weather}_*.jpg"
        rgb_files = list(rgb_path.glob(rgb_pattern))
        
        for rgb_file in tqdm(rgb_files, 
                            desc=f"Processing {camera} camera", 
                            leave=False):
            world_tick = int(rgb_file.stem.split('_')[-1])
            ticks_by_sensor[f'rgb_{camera.lower()}'].add(world_tick)
        sensor_pbar.update(1)
    
    # Find synchronized ticks
    all_ticks = set.union(*ticks_by_sensor.values())
    synchronized_ticks = {}
    
    for world_tick in tqdm(sorted(all_ticks), 
                          desc="Checking synchronization",
                          leave=True):
        has_all_data = all(world_tick in sensor_ticks 
                          for sensor_ticks in ticks_by_sensor.values())
        synchronized_ticks[world_tick] = has_all_data
        
        if not has_all_data:
            missing_sensors = [
                sensor for sensor, ticks in ticks_by_sensor.items()
                if world_tick not in ticks
            ]
            logging.warning(
                f"World tick {world_tick} missing data from: {', '.join(missing_sensors)}"
            )
    
    # Close main progress bar
    sensor_pbar.close()
    
    # Log summary
    complete_ticks = sum(synchronized_ticks.values())
    logging.info(f"\nSummary for User {user_num:03d}, {town}, {route}, {weather}:")
    logging.info(f"Total ticks found: {len(all_ticks)}")
    logging.info(f"Ticks with complete data: {complete_ticks}")
    logging.info(f"Ticks with missing data: {len(all_ticks) - complete_ticks}")
    
    # Also print summary to console
    print(f"\nSummary for User {user_num:03d}:")
    print(f"Total ticks found: {len(all_ticks)}")
    print(f"Ticks with complete data: {complete_ticks}")
    print(f"Ticks with missing data: {len(all_ticks) - complete_ticks}")
    
    return synchronized_ticks

def get_valid_world_ticks(
    base_path: str,
    user_num: int,
    town: str = "Town01",
    route: str = "R01",
    weather: str = "ClearNoon"
) -> List[int]:
    """
    Return a sorted list of world_ticks that have complete data.
    """
    synchronized_ticks = find_synchronized_ticks(
        base_path, user_num, town, route, weather
    )
    return sorted([
        tick for tick, is_complete in synchronized_ticks.items() 
        if is_complete
    ])

def process_multiple_users(
    base_path: str,
    user_nums: List[int],
    town: str = "Town01",
    route: str = "R01",
    weather: str = "ClearNoon"
) -> Dict[int, Dict[int, bool]]:
    results = {}
    
    for user_num in tqdm(user_nums, desc="Processing users", position=0):
        results[user_num] = find_synchronized_ticks(
            base_path, user_num, town, route, weather
        )
    
    return results

In [13]:
dataset_path_root = '/data-net/ted/users'
user_root_path = '/D019/Town01/R01'

res = get_valid_world_ticks(
    dataset_path_root, 19
)

Scanning sensors (User 019):   0%|          | 0/4 [00:00<?, ?it/s]

PermissionError: [Errno 13] Permission denied: '/data-net/ted/users/D019/Town01/R01/CB'